In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import kagglehub
import warnings

from sklearn.preprocessing import LabelEncoder, StandardScaler, OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import Ridge, Lasso
from sklearn.svm import SVR
from xgboost import XGBRegressor
from sklearn.utils import resample

import scipy.stats as stats
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.metrics import mean_absolute_percentage_error, median_absolute_error
from sklearn.model_selection import cross_val_score, learning_curve
import pickle

In [ ]:

pd.options.display.float_format = '{:.2f}'.format
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings('ignore')


df = pd.read_csv('/content/website_wata.csv')

In [ ]:
df.describe()

In [ ]:

print(f"Loaded {df.shape[0]} records and {df.shape[1]} columns.")

In [ ]:

df.info()

In [ ]:
df.dtypes

In [ ]:
df.head()

In [ ]:
df.tail()

In [ ]:
df.columns.tolist()

In [ ]:
print(df["Conversion Rate"].describe())

In [ ]:
print(df["Conversion Rate"].describe())

Separate majority and minority classes

In [ ]:
majority = df[df["Conversion Rate"] == 1.0]
minority = df[df["Conversion Rate"] < 1.0]

Oversampling the Minority Class

Upsample the minority class to match the size of the majority class



In [ ]:
minority_upsampled = resample(minority, replace=True, n_samples=len(majority), random_state=42)



Combine the majority class with the upsampled minority class


In [ ]:
df_balanced = pd.concat([majority, minority_upsampled])


Check the distribution of classes


In [ ]:
print(df_balanced["Conversion Rate"].value_counts())

Undersampling the Majority Class
Undersample the majority class to match the size of the minority class

In [ ]:
majority_sampled = majority.sample(n=len(minority), random_state=42)

Combine the undersampled majority class with the minority class

In [ ]:
df_balanced = pd.concat([majority_sampled, minority])

Check the distribution of classes

In [ ]:
print(df_balanced["Conversion Rate"].value_counts())

In [ ]:
df.head()

# **DATA PREPROCESSING**

In [ ]:
df.isnull().sum()

In [ ]:
df.isna().sum()

In [ ]:
duplicates = df.duplicated()
print(f"Number of duplicate rows: {duplicates.sum()}")

Suppose df["Conversion Rate"] is in percent (e.g. 34.0 for 34%)

In [ ]:
df["Conversion Rate"] = df["Conversion Rate"] / 100.0

To break the hard ceiling at 1.0, replace exact 1.0’s with a tiny epsilon below it:

This “jitter” gives the model some variation to learn from at the upper bound

In [ ]:
eps = 1e-3
df["Conversion Rate"] = df["Conversion Rate"].replace(1.0, 1.0 - eps)

#**VISUALIZATIONS**

Distribution of Page Views

In [ ]:
plt.figure(figsize=(10, 6))
sns.histplot(df['Page Views'], kde=True)
plt.title('Distribution of Page Views')
plt.xlabel('Page Views')
plt.ylabel('Frequency')
plt.show()

Distribution Of Traffic Source

In [ ]:
sns.countplot(x='Traffic Source',data=df)

Distribution Of Session Duration

In [ ]:
df['Session Duration'].plot(kind='hist')

Scatter plot of Session Duration vs. Conversion Rate

In [ ]:
plt.figure(figsize=(10, 6))
sns.scatterplot(x='Session Duration', y='Conversion Rate', data=df)
plt.title('Session Duration vs. Conversion Rate')
plt.xlabel('Session Duration (seconds)')
plt.ylabel('Conversion Rate')
plt.show()

In [ ]:
le = LabelEncoder()

df['Traffic Source'] = le.fit_transform(df['Traffic Source'])
sns.heatmap(df.corr(),annot=True)

#**CORRELATION HEATMAP**

In [ ]:
numeric_df = df.select_dtypes(include=[np.number])
plt.figure(figsize=(12, 8))
sns.heatmap(numeric_df.corr(), annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Correlation Heatmap')
plt.show()

In [ ]:
df.head()

summary of the dataset

In [ ]:
print(f"Loaded {df.shape[0]} records and {df.shape[1]} columns.")

Cleaned data

In [ ]:
df.to_csv("Cleaned_Data.csv", index=False)

In [ ]:
df

# **M0DEL SELECTION**

 Define Target And Independent Variable

In [ ]:

X = df.drop("Conversion Rate", axis=1)
y = df["Conversion Rate"]


Split data into training and testing sets (80% training, 20% testing)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

Initialize models

In [ ]:
models = {
    'Ridge Regression': Ridge(),
    'Lasso Regression': Lasso(),
    'Random Forest': RandomForestRegressor(),
    'Gradient Boosting': GradientBoostingRegressor(),
    'XGBoost' : XGBRegressor(),
    'Support Vector Regressor': SVR()
}

List to store results

In [ ]:
results = []

In [ ]:
for model_name, model in models.items():

    # Train the model
    model.fit(X_train, y_train)

    # Make predictions
    y_pred = model.predict(X_test)

    # Calculate metrics
    r2 = r2_score(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_test, y_pred)
    mape = mean_absolute_percentage_error(y_test, y_pred) * 100

    # Print results
    print(f"\n{model_name}:")
    print(f"R²: {r2:.4f}")
    print(f"MSE: {mse:.4f}")
    print(f"RMSE: {rmse:.4f}")
    print(f"MAE: {mae:.4f}")
    print(f"MAPE: {mape: .4f}%")

    results.append([
        model_name,
        round(r2, 4),
        round(mse, 4),
        round(rmse, 4),
        round(mae, 4),
        round(mape, 4)
    ])

Create Dataframe

In [ ]:
results_df = pd.DataFrame(results, columns=["Model", "R²", "MSE", "RMSE", "MAE", "MAPE (%)"])


Display table

In [ ]:
print(results_df)

Display as Markdown

In [ ]:
print(results_df.to_markdown(index=False))

# **MODEL PIPELINE AND TRAINING**

In [ ]:
numeric_features = ["Page Views", "Session Duration", "Bounce Rate", "Time on Page", "Previous Visits"]
categorical_features = ["Traffic Source"]

In [ ]:
preprocessor = ColumnTransformer([
    ("num", StandardScaler(), numeric_features),
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
])

In [ ]:

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# SELECTING RANDOM FOREST REGRESSOR MODEL

In [ ]:
print("\n Training Random Forest...")

rf_pipeline = Pipeline([
    ("preprocessing", preprocessor),
    ("regressor", RandomForestRegressor(random_state=42))
])

rf_pipeline.fit(X_train, y_train)
rf_pred = rf_pipeline.predict(X_test)

print("\n Trained Random Forest...")

# **MODEL EVALUATION**

Proportion of variance explained by the model (1.0 is perfect) (R2 Score)

In [ ]:
r2 = r2_score(y_test, rf_pred)
print(f"R2: {r2:.2f}")

Average squared difference between predicted and actual values (Mean Squared Error)

In [ ]:
mse = mean_squared_error(y_test, rf_pred)
print(f"MSE: {mse}")

The standard deviation of prediction errors, providing an interpretable error value in the same unit as the target variable (Root Mean Squared Error (RMSE))

In [ ]:
rmse = np.sqrt(mse)
print(f"RMSE: {rmse}")

The average squared difference between predicted and actual values (Root Mean Absolute Error)

In [ ]:
mae = mean_absolute_error(y_test, rf_pred)
print(f"MAE: {mae}")

Compute Mean Absolute Percentage Error (MAPE)


Measures average prediction error in percentage terms

In [ ]:
mape = np.mean(np.abs((y_test - rf_pred) / y_test)) * 100
print(f"MAPE: {mape:.2f}%")

In [ ]:

regressor = rf_pipeline.named_steps["regressor"]
if hasattr(regressor, "feature_importances_"):

    ohe = rf_pipeline.named_steps["preprocessing"].named_transformers_["cat"]
    onehot_labels = ohe.get_feature_names_out(categorical_features)
    feature_names = numeric_features + list(onehot_labels)

    importances = regressor.feature_importances_
    sorted_idx = np.argsort(importances)[::-1]

Feature Importance - Plot

In [ ]:
plt.figure(figsize=(10, 6))

sns.barplot(x=importances[sorted_idx], y=np.array(feature_names)[sorted_idx])
plt.title("Random Forest Regressor - Feature Importance")
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.tight_layout()
plt.show()

Cross Validation Score

In [ ]:
scores = cross_val_score(rf_pipeline, X, y, cv=5, scoring='r2')
print(f"Cross-validated R² scores: {scores}")
print(f"Average R²: {scores.mean():.3f}")

Average Cross Validation Scores

In [ ]:
print(f"Random Forest Average CV R²: {scores.mean():.3f}")

Adjusted R² penalizes excessive features to prevent overfitting

In [ ]:
n = len(y_test)

In [ ]:
p = X_test.shape[1]

adjusted_r2 = 1 - (1 - r2) * (n - 1) / (n - p - 1)

print(f"Adjusted R²: {adjusted_r2:.4f}")


Residual Analysis - Plot Residuals

Helps visualize errors to detect overfitting or underfitting

In [ ]:
residuals = y_test - rf_pred

plt.figure(figsize=(6, 4))
plt.scatter(y_pred, residuals, color='blue', alpha=0.5)
plt.axhline(y=0, color='red', linestyle='--')
plt.xlabel("Predicted Values")
plt.ylabel("Residuals")
plt.title("RF Regressor - Residual Distribution")
plt.show()

 Residual Normality Test (Shapiro-Wilk Test)

Ensures errors are normally distributed (Ideal for linear regression models)

In [ ]:
stat, p_value = stats.shapiro(residuals)
print(f"P-value: {p_value:.5f} (Should be > 0.05 for normal distribution)")

In [ ]:
df.dtypes

Actual vs Predicted Plot - This plot compares predicted values with actual values

In [ ]:
plt.figure(figsize=(6, 4))

sns.scatterplot(x=y_test, y=rf_pred, alpha=0.6)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
plt.title("RF Regressor - Actual vs. Predicted")
plt.xlabel("Actual")
plt.ylabel("Predicted")
plt.tight_layout()
plt.show()

Quantile Error Metrics - for skewed data, use quantile loss or metrics like median absolute error

In [ ]:
print(f"Median Absolute Error: {median_absolute_error(y_test, rf_pred):.3f}")

Prediction Error Distribution - shows distribution of residuals (errors)

In [ ]:
errors = y_test - rf_pred
sns.histplot(errors, bins=30, kde=True)
plt.title("RF Regressor - Distribution of Prediction Errors")
plt.xlabel("Error")
plt.show()

Compute learning curve data

In [ ]:
train_sizes, train_scores, test_scores = learning_curve(
    rf_pipeline, X, y, cv=5, scoring='r2',
    train_sizes=np.linspace(0.1, 1.0, 5), random_state=42
)



Compute mean and standard deviation


In [ ]:
train_scores_mean = train_scores.mean(axis=1)
test_scores_mean = test_scores.mean(axis=1)
train_scores_std = train_scores.std(axis=1)
test_scores_std = test_scores.std(axis=1)

Learning Curve - Visualizes model performance as training size increases (helps detect overfitting/underfitting)


Plotting the learning curve


In [ ]:
plt.figure(figsize=(8, 6))
plt.plot(train_sizes, train_scores_mean, 'o-', color="blue", label="Training score")
plt.plot(train_sizes, test_scores_mean, 'o-', color="green", label="Cross-validation score")


Shaded std deviation

In [ ]:
plt.fill_between(
    train_sizes, train_scores_mean - train_scores_std,
    train_scores_mean + train_scores_std, alpha=0.1, color="blue"
)

plt.fill_between(
    train_sizes, test_scores_mean - test_scores_std,
    test_scores_mean + test_scores_std, alpha=0.1, color="green"
)


Labels and legend

In [ ]:
plt.title("📈 Learning Curve - Random Forest")
plt.xlabel("Training Set Size")
plt.ylabel("R² Score")
plt.legend(loc="best")
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# Learning Curve - Visualizes model performance as training size increases (helps detect overfitting/underfitting)


plt.figure(figsize=(8, 6))
plt.plot(train_sizes, train_scores_mean, 'o-', color="blue", label="Training score")
plt.plot(train_sizes, test_scores_mean, 'o-', color="green", label="Cross-validation score")


plt.fill_between(
    train_sizes, train_scores_mean - train_scores_std,
    train_scores_mean + train_scores_std, alpha=0.1, color="blue"
)

plt.fill_between(
    train_sizes, test_scores_mean - test_scores_std,
    test_scores_mean + test_scores_std, alpha=0.1, color="green"
)


plt.title("📈 Learning Curve - Random Forest")
plt.xlabel("Training Set Size")
plt.ylabel("R² Score")
plt.legend(loc="best")
plt.grid(True)
plt.tight_layout()
plt.show()